In [1]:
import gc
import numpy as np
import os
import re
import sys

import pandas as pd
import tensorflow_data_validation as tfdv

#VERSION = "20221025"
DATA_DIR = r"Z:\13.Data Science\Projects\2022_GI_Xsell\Data"
PROJECT_DIR = "../"
RAW_DATA_DIR = os.path.join(DATA_DIR, "raw")
PROCESSED_DATA_DIR = os.path.join(DATA_DIR, "processed")
META_DATA_DIR = os.path.join(DATA_DIR, "meta_and_reference_data")
VER_1025_DIR = os.path.join(RAW_DATA_DIR, "20221025")
VER_1118_DIR = os.path.join(RAW_DATA_DIR, "20221118")
VER_1122_DIR = os.path.join(RAW_DATA_DIR, "20221122")
VER_1212_DIR = os.path.join(RAW_DATA_DIR, "20221212")
GI_DATA_DIR = os.path.join(RAW_DATA_DIR, "GI")
PA_DATA_DIR = os.path.join(GI_DATA_DIR, "Travel", "202210")
sys.path.insert(0, DATA_DIR)
sys.path.insert(0, PROJECT_DIR)

from datetime import datetime
from dateutil.relativedelta import relativedelta

# from src.utils.common import PLAN_TYPE_LST, WGP_LST, INFORCE_STATUS_LST, TRAVEL_SINGLE_MULTI_TRIP_DICT
# from src.utils.helper_function import find_plan_type, one_hot

In [2]:
pd.set_option('display.width', None)
pd.set_option('max_colwidth', None)
# pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)

In [3]:
BASESTAT_FILE = os.path.join(PA_DATA_DIR, "basestat.parquet")
CLAMBASE_FILE = os.path.join(PA_DATA_DIR, "clambase_df.parquet")
PREMBASE_FILE = os.path.join(PA_DATA_DIR, "prembase.parquet")
CLIENT_DATA_MAPPING_FILE = os.path.join(PROCESSED_DATA_DIR, "processed_client_data.parquet")

In [4]:
basestat_df = pd.read_parquet(BASESTAT_FILE)


In [5]:
basestat_df.sample(3)

,contrnb,CCDATE,RSKNO,DTEEFF,CNTTYPE,OCCDATE,COWNNUM,AGNTNUM,CNTBRANCH,autorenew,ZDMIND,bulk,bulk_name,nbrn,tranat,product,bulk_name1,CLTSEX,INSURED,ZPLANCDE,trlarea,znolive,plan1,duration,lastznolive,plan2,plan3,ChinaCard,EnhancedPA,duration_grp,producer,rmisagent,channel,lifeagent,product2,gap_trandate_ccdate,gap_trandate_ccdate_gp,znolive_gp,age_gp,loyalty,loyalty_cat,insage_smu,inssex_smu,insage_smu_gp,disc_gp,gwp,nwp,gep,nep,yduration,disc,ypol,ypol_actual,no_pol,no_rsk,commis,commis_n,nec,if_pol,if_rsk,ExpectedCost,TechnicalPremium,MarketingPremium,GWP_initial,APTP_gp,chgbal_g,chgbal_n,paymnt_g,paymnt_n,nb_clm,incur_g,incur_n,ultincur_n,ult_clm,lrg_incur_g,lrg_incur_n,lrg_ultincur,if_znolive,Agg_CA_nb_clm,Agg_EA_nb_clm,Agg_ME_nb_clm,Agg_PA_nb_clm,Agg_PB_nb_clm,Agg_TD_nb_clm,Agg_TP_nb_clm,Agg_CA_ultclm,Agg_EA_ultclm,Agg_ME_ultclm,Agg_PA_ultclm,Agg_PB_ultclm,Agg_TD_ultclm,Agg_TP_ultclm,Agg_CA_incur_g,Agg_EA_incur_g,Agg_ME_incur_g,Agg_PA_incur_g,Agg_PB_incur_g,Agg_TD_incur_g,Agg_TP_incur_g,Agg_CA_incur_n,Agg_EA_incur_n,Agg_ME_incur_n,Agg_PA_incur_n,Agg_PB_incur_n,Agg_TD_incur_n,Agg_TP_incur_n,Agg_CA_ultincur,Agg_EA_ultincur,Agg_ME_ultincur,Agg_PA_ultincur,Agg_PB_ultincur,Agg_TD_ultincur,Agg_TP_ultincur,period,total_yduration,no_znolive
2888009,25446674,20180716.0,1.0,20180716.0,STR,20180716.0,2688546,E81403,20,0.0,N,N,N,NB,NWBS,TravelSurance (single trip),N,M,NaN,I2,2,1.0,Single,5.0,1.0,Area 2,Self Only,N,NA,3. 4-5 Days,HSBC - Dummy,637(SURANCE)-RDP,Banca - HSBC,0.0,TravelSurance,25.0,6. 21 to 25 days,1. 1 person,5. Age 51 - 60,-0.0,Single,NaN,NaN,NaN,1. Discount = 1% to 10%,34.70,34.422400,34.70,34.422400,5.0,3.85,0.013689,1.0,1.0,1.0,10.41,10.41,10.41,0.0,0.0,0.0,0.0,0.0,34.70,12. unknown,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,20180101.0,5.0,1.0
2829501,25414453,20180518.0,1.0,20180518.0,STR,20180518.0,2580581,E04432,20,0.0,N,N,N,NB,NWBS,TravelSurance (single trip),N,M,NaN,I1,1,1.0,Single,6.0,1.0,Area 1,Self Only,N,NA,4. 6-7 Days,HSBC - Dummy,637STAFF(SURANCE)-NET,Banca - HSBC,0.0,TravelSurance,0.0,1. 0 days or les,1. 1 person,5. Age 51 - 60,-0.0,Single,NaN,NaN,NaN,5. Discount = 26% to 30%,18.17,18.024640,18.17,18.024640,6.0,7.78,0.016427,1.0,1.0,1.0,0.00,0.00,0.00,0.0,0.0,0.0,0.0,0.0,18.17,12. unknown,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,20180101.0,6.0,1.0
4968807,26001529,20220908.0,2.0,20220908.0,STQ,20220908.0,6090210,E82391,20,0.0,N,N,N,NB,NWBS,Single Trip 2018,N,M,NaN,*W,W,NaN,Single,11.0,NaN,Asia,Basic,N,NA,6. 11-14 Days,HSBC - Dummy,637(SURANCE)-MBL,Banca - HSBC,0.0,TravelSurance - 2018,0.0,1. 0 days or les,1. 1 person,3. Age 31 - 40,-0.0,Single,NaN,NaN,NaN,2. Discount = 11% to 15%,237.48,234.653988,237.48,234.653988,11.0,41.91,0.000000,0.0,0.0,1.0,71.24,71.24,71.24,0.0,0.0,0.0,0.0,0.0,237.48,12. unknown,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,20220101.0,0.0,0.0


In [6]:
client_data_df = pd.read_parquet(CLIENT_DATA_MAPPING_FILE)

In [7]:
client_data_df.sample(3)

,CLNTNUM,VALIDFLAG,CLTTYPE,SECUITYNO,CLTSEX,OCCPCODE,CLTDOB,START_DATE,MARRYD,NATLTY,modified_SECUITYNO
5056955,1191939,1,P,K748122(5),F,None,19731125.0,20180126.0,None,None,K748122
2390018,O8958818,1,P,E948542(6),F,D209,19570215.0,99999999.0,M,None,E948542
453083,0466041,1,P,G283655(5),M,None,19651116.0,20091007.0,None,HK,G283655


In [9]:
# strip COWNNUM and CLNTNUM
# kelvin change from client_data_df to basestat_df
#basestat_df["COWNNUM"] = client_data_df["COWNNUM"].str.strip()
basestat_df["COWNNUM"] = basestat_df["COWNNUM"].str.strip()
client_data_df["CLNTNUM"] = client_data_df["CLNTNUM"].str.strip()

In [10]:
# count how many unique COWNNUM in basestat_df that is not in CLNTNUM of client_data_df
basestat_df[~basestat_df.COWNNUM.isin(client_data_df.CLNTNUM)].COWNNUM.nunique()

32227

In [11]:
# extract the COWNNUM in basestat_df that is not in CLNTNUM of client_data_df and return to a new df
basestat_not_in_clientbase_df = basestat_df[basestat_df.COWNNUM.isin(basestat_df[~basestat_df.COWNNUM.isin(client_data_df.CLNTNUM)].COWNNUM)]

In [12]:
basestat_not_in_clientbase_df.sample(10)

,contrnb,CCDATE,RSKNO,DTEEFF,CNTTYPE,OCCDATE,COWNNUM,AGNTNUM,CNTBRANCH,autorenew,ZDMIND,bulk,bulk_name,nbrn,tranat,product,bulk_name1,CLTSEX,INSURED,ZPLANCDE,trlarea,znolive,plan1,duration,lastznolive,plan2,plan3,ChinaCard,EnhancedPA,duration_grp,producer,rmisagent,channel,lifeagent,product2,gap_trandate_ccdate,gap_trandate_ccdate_gp,znolive_gp,age_gp,loyalty,loyalty_cat,insage_smu,inssex_smu,insage_smu_gp,disc_gp,gwp,nwp,gep,nep,yduration,disc,ypol,ypol_actual,no_pol,no_rsk,commis,commis_n,nec,if_pol,if_rsk,ExpectedCost,TechnicalPremium,MarketingPremium,GWP_initial,APTP_gp,chgbal_g,chgbal_n,paymnt_g,paymnt_n,nb_clm,incur_g,incur_n,ultincur_n,ult_clm,lrg_incur_g,lrg_incur_n,lrg_ultincur,if_znolive,Agg_CA_nb_clm,Agg_EA_nb_clm,Agg_ME_nb_clm,Agg_PA_nb_clm,Agg_PB_nb_clm,Agg_TD_nb_clm,Agg_TP_nb_clm,Agg_CA_ultclm,Agg_EA_ultclm,Agg_ME_ultclm,Agg_PA_ultclm,Agg_PB_ultclm,Agg_TD_ultclm,Agg_TP_ultclm,Agg_CA_incur_g,Agg_EA_incur_g,Agg_ME_incur_g,Agg_PA_incur_g,Agg_PB_incur_g,Agg_TD_incur_g,Agg_TP_incur_g,Agg_CA_incur_n,Agg_EA_incur_n,Agg_ME_incur_n,Agg_PA_incur_n,Agg_PB_incur_n,Agg_TD_incur_n,Agg_TP_incur_n,Agg_CA_ultincur,Agg_EA_ultincur,Agg_ME_ultincur,Agg_PA_ultincur,Agg_PB_ultincur,Agg_TD_ultincur,Agg_TP_ultincur,period,total_yduration,no_znolive
5080213,Z1650950,20221014.0,1.0,20221015.0,STJ,20221014.0,1733950,00411,10,0.0,N,N,N,NB,ENDO,SmartTraveller Plus (Single),N,NaN,S0,Q4,9,7.0,Single,NaN,7.0,Prestige,Insured Only,N,Y,6. 11-14 Days,Assurance Appraisal Ltd,ASSURANCE APPRAISAL LIMITED,Agents & Broker,0.0,SmartTraveller Plus,22.0,6. 21 to 25 days,3. >2 people,9. unknown,-0.0,Single,NaN,NaN,NaN,0. No Discount,0.00,0.000000,0.000000,0.000000,11.0,0.00,0.000000,0.000000,0.0,0.0,0.00,0.00,0.000000,0.0,0.0,0.000000,0.000000,0.0,0.00,12. unknown,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000,0.0,0.0,20220101.0,77.0,0.0
4965079,25999658,20220814.0,2.0,20220814.0,STQ,20220814.0,6086808,E82391,20,0.0,N,N,N,NB,NWBS,Single Trip 2018,N,F,NaN,*Y,Y,NaN,Single,22.0,NaN,Worldwide,Basic,N,NA,7. 15-30 Days,HSBC - Dummy,637(SURANCE)-MBL,Banca - HSBC,0.0,TravelSurance - 2018,1.0,2. 1 to 5 days,1. 1 person,2. Age 17 - 30,-0.0,Single,NaN,NaN,NaN,2. Discount = 11% to 15%,470.15,464.555215,470.150000,464.555215,22.0,82.97,0.000000,0.000000,0.0,1.0,141.05,141.05,141.050000,0.0,0.0,0.000000,0.000000,0.0,470.15,12. unknown,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000,0.0,0.0,20220101.0,0.0,0.0
5077832,Z1643835,20221011.0,1.0,20221011.0,STJ,20221011.0,1737744,74857,10,0.0,N,N,N,NB,NWBS,SmartTraveller Plus (Single),N,F,S0,Q4,9,1.0,Single,5.0,1.0,Prestige,Insured Only,N,Y,3. 4-5 Days,Alpha Agency,LAM WAI TAT,LP,1.0,SmartTraveller Plus,7.0,3. 6 to 10 days,1. 1 person,2. Age 17 - 30,-0.0,Single,NaN,NaN,NaN,3. Discount = 16% to 20%,244.00,241.096400,244.000000,241.096400,5.0,61.00,0.013689,1.000000,1.0,1.0,61.00,61.00,61.000000,0.0,0.0,99.187177,261.497715,305.0,244.00,5. -5% to -15%,6000.0,6000.0,0.0,0.0,1.0,6000.0,6000.0,4128.942,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,6000.0,0.0,0.0,0.0,0.0,0.0,0.0,6000.0,0.0,0.0,0.0,0.0,0.0,0.0,4128.942,0.0,0.0,20220101.0,5.0,1.0
5038939,Z1378421,20220628.0,1.0,20220628.0,STJ,20220628.0,1710367,71301,10,0.0,N,N,N,NB,NWBS,SmartTraveller Plus (Single),N,F,S0,O4,9,1.0,Single,46.0,1.0,Advance,Insured Only,N,N,8. 31-60 Days,AFFINITY SUNNY TSE AGENCY,CHAN LAI CHUN CLARA,LP,1.0,SmartTraveller Plus,0.0,1. 0 days or les,1. 1 person,4. Age 41 - 50,-0.0,Single,NaN,NaN,NaN,1. Discount = 1% to 10%,1004.40,992.447640,1004.400000,992.447640,1.0,111.60,0.125941,0.021739,1.0,1.0,251.10,251.10,251.100000,0.0,0.0,591.972691,1280.268723,1116.0,1004.40,4. -15% to -25%,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000,0.0,0.0,0.0,0.0,

In [13]:
# group by channel and count the number of unique contrnb
basestat_not_in_clientbase_df.groupby("channel").contrnb.nunique()


channel
AXA Offline Direct       26
AXA Online Direct     10691
Agents & Broker        5949
Banca - HSBC          10763
Citi                    273
JLT (OSC)                91
LP                     5093
Name: contrnb, dtype: int64

In [14]:
# group by channel and count the number of unique contrnb
basestat_not_in_clientbase_df.groupby("channel").COWNNUM.nunique()


channel
AXA Offline Direct       24
AXA Online Direct     10691
Agents & Broker        5921
Banca - HSBC          10149
Citi                    273
JLT (OSC)                85
LP                     5091
Name: COWNNUM, dtype: int64

In [15]:
basestat_not_in_clientbase_df.drop_duplicates("COWNNUM").channel.value_counts(normalize=True, dropna=False)

AXA Online Direct     0.331730
Banca - HSBC          0.314912
Agents & Broker       0.183722
LP                    0.157782
Citi                  0.008471
JLT (OSC)             0.002606
AXA Offline Direct    0.000745
NaN                   0.000031
Name: channel, dtype: float64

In [16]:
basestat_not_in_clientbase_df.drop_duplicates("COWNNUM").channel.value_counts(normalize=False, dropna=False)

AXA Online Direct     10691
Banca - HSBC          10149
Agents & Broker        5921
LP                     5085
Citi                    273
JLT (OSC)                84
AXA Offline Direct       24
NaN                       1
Name: channel, dtype: int64

In [17]:
# count number of COWNNUM is None
basestat_not_in_clientbase_df[basestat_not_in_clientbase_df.COWNNUM.isna()]


,contrnb,CCDATE,RSKNO,DTEEFF,CNTTYPE,OCCDATE,COWNNUM,AGNTNUM,CNTBRANCH,autorenew,ZDMIND,bulk,bulk_name,nbrn,tranat,product,bulk_name1,CLTSEX,INSURED,ZPLANCDE,trlarea,znolive,plan1,duration,lastznolive,plan2,plan3,ChinaCard,EnhancedPA,duration_grp,producer,rmisagent,channel,lifeagent,product2,gap_trandate_ccdate,gap_trandate_ccdate_gp,znolive_gp,age_gp,loyalty,loyalty_cat,insage_smu,inssex_smu,insage_smu_gp,disc_gp,gwp,nwp,gep,nep,yduration,disc,ypol,ypol_actual,no_pol,no_rsk,commis,commis_n,nec,if_pol,if_rsk,ExpectedCost,TechnicalPremium,MarketingPremium,GWP_initial,APTP_gp,chgbal_g,chgbal_n,paymnt_g,paymnt_n,nb_clm,incur_g,incur_n,ultincur_n,ult_clm,lrg_incur_g,lrg_incur_n,lrg_ultincur,if_znolive,Agg_CA_nb_clm,Agg_EA_nb_clm,Agg_ME_nb_clm,Agg_PA_nb_clm,Agg_PB_nb_clm,Agg_TD_nb_clm,Agg_TP_nb_clm,Agg_CA_ultclm,Agg_EA_ultclm,Agg_ME_ultclm,Agg_PA_ultclm,Agg_PB_ultclm,Agg_TD_ultclm,Agg_TP_ultclm,Agg_CA_incur_g,Agg_EA_incur_g,Agg_ME_incur_g,Agg_PA_incur_g,Agg_PB_incur_g,Agg_TD_incur_g,Agg_TP_incur_g,Agg_CA_incur_n,Agg_EA_incur_n,Agg_ME_incur_n,Agg_PA_incur_n,Agg_PB_incur_n,Agg_TD_incur_n,Agg_TP_incur_n,Agg_CA_ultincur,Agg_EA_ultincur,Agg_ME_ultincur,Agg_PA_ultincur,Agg_PB_ultincur,Agg_TD_ultincur,Agg_TP_ultincur,period,total_yduration,no_znolive
0,00301996,NaN,2.0,NaN,NaN,NaN,None,None,NaN,NaN,NaN,NaN,None,NaN,NaN,None,None,NaN,NaN,None,None,NaN,NaN,NaN,NaN,None,None,NaN,NaN,NaN,None,None,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0000,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000,0.0,0.0,0.0,0.000,0.0000,0.0,20140101.0,0.0,0.0
1,00307066,NaN,2.0,NaN,NaN,NaN,None,None,NaN,NaN,NaN,NaN,None,NaN,NaN,None,None,NaN,NaN,None,None,NaN,NaN,NaN,NaN,None,None,NaN,NaN,NaN,None,None,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0000,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000,0.0,0.0,0.0,0.000,0.0000,0.0,20140101.0,0.0,0.0
2,00310357,NaN,2.0,NaN,NaN,NaN,None,None,NaN,NaN,NaN,NaN,None,NaN,NaN,None,None,NaN,NaN,None,None,NaN,NaN,NaN,NaN,None,None,NaN,NaN,NaN,None,None,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0000,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000,0.0,0.0,0.0,0.000,0.0000,0.0,20140101.0,0.0,0.0
3,00314982,NaN,2.0,NaN,NaN,NaN,None,None,NaN,NaN,NaN,NaN,None,NaN,NaN,None,None,NaN,NaN,None,None,NaN,NaN,NaN,NaN,None,None,NaN,NaN,NaN,None,None,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0000,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000,0.0,0.0,0.0,0.000,0.0000,0.0,20140101.0,0.0,0.0
4,00319036,NaN,2.0,NaN,NaN,NaN,None,None,NaN,NaN,NaN,NaN,None,NaN,NaN,None,None,NaN,NaN,None,None,NaN,NaN,NaN,NaN,None,None,NaN,NaN,NaN,None,None,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0000,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000,0.0,0.0,0.0,0.000,0.0000,0.0,20140101.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,..

In [28]:
basestat_not_in_clientbase_df[basestat_not_in_clientbase_df["COWNNUM"]=="1737744"]

,contrnb,CCDATE,RSKNO,DTEEFF,CNTTYPE,OCCDATE,COWNNUM,AGNTNUM,CNTBRANCH,autorenew,ZDMIND,bulk,bulk_name,nbrn,tranat,product,bulk_name1,CLTSEX,INSURED,ZPLANCDE,trlarea,znolive,plan1,duration,lastznolive,plan2,plan3,ChinaCard,EnhancedPA,duration_grp,producer,rmisagent,channel,lifeagent,product2,gap_trandate_ccdate,gap_trandate_ccdate_gp,znolive_gp,age_gp,loyalty,loyalty_cat,insage_smu,inssex_smu,insage_smu_gp,disc_gp,gwp,nwp,gep,nep,yduration,disc,ypol,ypol_actual,no_pol,no_rsk,commis,commis_n,nec,if_pol,if_rsk,ExpectedCost,TechnicalPremium,MarketingPremium,GWP_initial,APTP_gp,chgbal_g,chgbal_n,paymnt_g,paymnt_n,nb_clm,incur_g,incur_n,ultincur_n,ult_clm,lrg_incur_g,lrg_incur_n,lrg_ultincur,if_znolive,Agg_CA_nb_clm,Agg_EA_nb_clm,Agg_ME_nb_clm,Agg_PA_nb_clm,Agg_PB_nb_clm,Agg_TD_nb_clm,Agg_TP_nb_clm,Agg_CA_ultclm,Agg_EA_ultclm,Agg_ME_ultclm,Agg_PA_ultclm,Agg_PB_ultclm,Agg_TD_ultclm,Agg_TP_ultclm,Agg_CA_incur_g,Agg_EA_incur_g,Agg_ME_incur_g,Agg_PA_incur_g,Agg_PB_incur_g,Agg_TD_incur_g,Agg_TP_incur_g,Agg_CA_incur_n,Agg_EA_incur_n,Agg_ME_incur_n,Agg_PA_incur_n,Agg_PB_incur_n,Agg_TD_incur_n,Agg_TP_incur_n,Agg_CA_ultincur,Agg_EA_ultincur,Agg_ME_ultincur,Agg_PA_ultincur,Agg_PB_ultincur,Agg_TD_ultincur,Agg_TP_ultincur,period,total_yduration,no_znolive
5077832,Z1643835,20221011.0,1.0,20221011.0,STJ,20221011.0,1737744,74857,10,0.0,N,N,N,NB,NWBS,SmartTraveller Plus (Single),N,F,S0,Q4,9,1.0,Single,5.0,1.0,Prestige,Insured Only,N,Y,3. 4-5 Days,Alpha Agency,LAM WAI TAT,LP,1.0,SmartTraveller Plus,7.0,3. 6 to 10 days,1. 1 person,2. Age 17 - 30,-0.0,Single,NaN,NaN,NaN,3. Discount = 16% to 20%,244.0,241.0964,244.0,241.0964,5.0,61.0,0.013689,1.0,1.0,1.0,61.0,61.0,61.0,0.0,0.0,99.187177,261.497715,305.0,244.0,5. -5% to -15%,6000.0,6000.0,0.0,0.0,1.0,6000.0,6000.0,4128.942,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,6000.0,0.0,0.0,0.0,0.0,0.0,0.0,6000.0,0.0,0.0,0.0,0.0,0.0,0.0,4128.942,0.0,0.0,20220101.0,5.0,1.0


In [29]:
client_data_df[client_data_df["CLNTNUM"]=="1737744"]

,CLNTNUM,VALIDFLAG,CLTTYPE,SECUITYNO,CLTSEX,OCCPCODE,CLTDOB,START_DATE,MARRYD,NATLTY,modified_SECUITYNO


In [18]:
# output to parquet
basestat_not_in_clientbase_df.to_parquet(os.path.join(PROCESSED_DATA_DIR, "basestat_not_in_clientbase_df.parquet"))

In [19]:
# get contrnb COWNNUM and channel from basestat_not_in_clientbase_df
theCols=basestat_not_in_clientbase_df[["contrnb", "COWNNUM", "channel"]]

In [20]:
# get non none contrnb COWNNUM and channel from basestat_not_in_clientbase_df
theCols=basestat_not_in_clientbase_df[~basestat_not_in_clientbase_df.COWNNUM.isna()][["contrnb", "COWNNUM", "channel"]]


In [21]:
theCols.value_counts(dropna=False)

contrnb   COWNNUM  channel           
57352519  6095528  Banca - HSBC          8
Z1480125  1717144  Agents & Broker       6
Z1481032  1728330  Agents & Broker       6
27649871  6090370  Banca - HSBC          6
27649460  6086011  Banca - HSBC          6
                                        ..
Z1384642  1725004  LP                    1
Z1384640  1724607  AXA Online Direct     1
Z1384639  1724610  AXA Online Direct     1
Z1384638  1724609  LP                    1
Z1942631  1748097  AXA Offline Direct    1
Length: 32887, dtype: int64

In [22]:
# compression_opts = dict(method='zip',
#                         archive_name='basestat_not_in_clientbase_df.csv')  
# theCols.to_csv('basestat_not_in_clientbase_df.zip', index=False,
#           compression=compression_opts)

In [23]:
theCols.drop_duplicates("contrnb").to_csv(r'Z:\13.Data Science\Projects\2022_GI_Xsell\Output\list\client_num_in_basestat_not_in_client_data.csv', index=False,)